# A3 DQN core + baselines + strategy 3

The shared DQN code is in core/ and the curriculum (strategy 3) is in strategies/curriculum.py. This notebook runs the experiments and makes the plots. Training cells skip runs that already finished so they can just be rerun.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from core.config import Cfg
from core.env import VAL_LENS
from train import launch, run_name

SEEDS = [0, 1, 2, 3, 4]
PROCS = 10  #parallel runs, 1 cpu thread each

## Setup

Everything below is the same for all methods so the only thing that changes between them is how the pole length gets picked each episode.

The network is the standard QNetwork from test_script.py. It's trained with double DQN using Huber loss and Adam, with one update every 4 env steps and a hard target network copy every 1000 env steps. Epsilon goes down linearly over the first 20% of the steps. The settings come from the two pilots below.

The reward is +1 per step minus a penalty for pole angle and cart position: $r = 1 - 0.5|\theta|/\theta_{max} - 0.5|x|/x_{max}$

There is no 500 step cutoff. Train episodes stop at 1000 steps but that's not stored as done so Q still bootstraps past it.

Every 10k steps the greedy policy is run on our own val lengths (the bin centres, not the test lengths) with normal dynamics and fixed seeds. Val score is the shaped return over 1000 steps divided by 1000 so 1 means it balanced perfectly the whole time. The checkpoint with the best mean val score is the one that gets saved.

In [ ]:
#default hyperparams, table for the methods section
pd.Series(Cfg().dict(), name='value').to_frame()

## Pilot

Small lr sweep on the uniform baseline to pick the lr and see how many steps are needed. Seeds 100 and 101 so they don't overlap with the real runs. Only the val score is used for this, not the test script.

In [ ]:
#pilot 1 used the first settings: update every step + soft tgt update
OLD = ['steps=400000', 'train_freq=1', 'tau=0.005', 'tgt_every=1']
PILOT_LRS = [1e-3, 5e-4, 2.5e-4]

pilot = [('uniform', s, [f'lr={lr}'] + OLD, f'_lr{lr:g}', 'pilot') for lr in PILOT_LRS for s in (100, 101)]
launch(pilot, PROCS)

In [ ]:
def load_val(names, root='.'):
    #val logs of several runs stacked
    dfs = []
    for n in names:
        df = pd.read_csv(f'{root}/runs/{n}/val.csv')
        df['run'] = n
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def plot_val(groups, root='.', ax=None):
    #mean +- std val score over seeds per group
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    for label, names in groups.items():
        g = load_val(names, root).groupby('step')['score']
        m, s = g.mean(), g.std().fillna(0)
        ax.plot(m.index, m, label=label)
        ax.fill_between(m.index, m - s, m + s, alpha=0.2)
    ax.set_xlabel('env steps')
    ax.set_ylabel('val score')
    ax.legend()
    ax.grid(alpha=0.3)
    return ax


def best_rows(names, root='.'):
    #val row of the saved ckpt per run
    df = load_val(names, root)
    return df.loc[df.groupby('run')['score'].idxmax()]

In [ ]:
pilot_groups = {f'lr={lr:g}': [run_name('uniform', s, f'_lr{lr:g}') for s in (100, 101)] for lr in PILOT_LRS}
plot_val(pilot_groups, 'pilot').set_title('pilot 1: uniform baseline, mean +- std over 2 seeds')
plt.show()

#best ckpt per run, where in training it happened
pd.concat({k: best_rows(v, 'pilot')[['run', 'step', 'score']] for k, v in pilot_groups.items()})

## Pilot 2

Pilot 1 was really unstable. With lr 1e-3 and 5e-4 the Q values blew up (one run went to 1e10 while the max possible is 100 with $\gamma = 0.99$ and reward $\leq 1$) and the agent ended up falling in about 10 steps. Even 2.5e-4 had big drops and one seed ended with Q values around 170. Part of the problem is doing an update every env step. Also without the 500 cutoff a good agent hardly ever fails so there aren't many failures left in the buffer.

So pilot 2 keeps lr 2.5e-4 and switches to more standard DQN settings: one update every 4 steps and a hard target copy every 1000 env steps (Mnih et al. 2015). On top of that it tries $\gamma = 0.98$ (smaller Q values) and a 400k buffer (keeps all the early failures).

In [ ]:
STD = ['lr=0.00025', 'steps=400000', 'train_freq=4', 'tau=1', 'tgt_every=250']
PILOT2 = {'_std': STD,
          '_g98': STD + ['gamma=0.98'],
          '_buf400': STD + ['buf=400000'],
          '_g98buf400': STD + ['gamma=0.98', 'buf=400000']}

launch([('uniform', s, kv, tag, 'pilot') for tag, kv in PILOT2.items() for s in (100, 101)], PROCS)

In [ ]:
import torch
from test_script import QNetwork


def q_upright(run, root='pilot'):
    #max Q of the final net at the upright state, real max is 1 / (1 - gamma)
    q = QNetwork(4, 2)
    q.load_state_dict(torch.load(f'{root}/runs/{run}/final.pth'))
    return q(torch.zeros(1, 4)).max().item()


pilot2_groups = {tag[1:]: [run_name('uniform', s, tag) for s in (100, 101)] for tag in PILOT2}
plot_val(pilot2_groups, 'pilot').set_title('pilot 2: uniform baseline, mean +- std over 2 seeds')
plt.show()

#stability after 200k steps for both pilots
rows = []
for k, names in {**pilot_groups, **pilot2_groups}.items():
    df = load_val(names, 'pilot')
    late = df[df.step > 200_000]
    rows.append({'cfg': k, 'best': df.groupby('run')['score'].max().mean(),
                 'late_mean': late['score'].mean(), 'late_min': late['score'].min(),
                 'q_upright': np.median([q_upright(n) for n in names])})

pd.DataFrame(rows).round(3)

std wins. It has the highest late mean (0.88 and 0.87 for the 2 seeds) and no collapse, and Q at the upright state is about 95 which is close to the real max of 100. $\gamma = 0.98$ and the big buffer both made it less stable. So the defaults in core/config.py are now lr 2.5e-4, one update every 4 steps, hard target copy every 1000 env steps, $\gamma = 0.99$ and 400k steps per run. It's only 2 seeds per setting so this is just for picking settings and not a result.

## Baselines

fixed: always length 0.5 (normal cartpole)

uniform: random length between 0.4 and 1.8 every episode. This is the main one we compare against.

In [ ]:
BASE = []  #defaults in core/config.py are already the pilot 2 std settings
launch([(m, s, BASE, '', '.') for m in ('fixed', 'uniform') for s in SEEDS], PROCS)

## Strategy 3: curriculum

0.4 to 1.8 is split into $K = 10$ bins and the bin centres are the val lengths. After every val round each bin gets a regret of 1 - val score. The bins are ranked on regret (worst bin is rank 1) and sampled with

$$p_i = (1-\lambda)\frac{h_i}{\sum_j h_j} + \frac{\lambda}{K}, \qquad h_i = \text{rank}_i^{-1/T}$$

This is the rank prioritisation from prioritized level replay (Jiang et al. 2021). The $\lambda/K$ part mixes in uniform sampling ($\lambda = 0.2$) so the easy lengths don't get forgotten. Every bin gets rescored each round so the staleness part from PLR isn't needed. Inside a bin the length is uniform so it doesn't only train on the val lengths.

In [ ]:
launch([('curriculum', s, BASE, '', '.') for s in SEEDS], PROCS)

## Results on the val lengths

In [ ]:
METHODS = ['fixed', 'uniform', 'curriculum']
groups = {m: [run_name(m, s) for s in SEEDS] for m in METHODS}
plot_val(groups).set_title('val score during training, mean +- std over 5 seeds')
plt.show()

In [ ]:
#per len val score of the saved ckpt
cols = [f's_{l:.2f}' for l in VAL_LENS]
x = np.arange(len(VAL_LENS))
w = 0.8 / len(METHODS)

fig, ax = plt.subplots(figsize=(10, 4))

for i, m in enumerate(METHODS):
    b = best_rows(groups[m])[cols]
    ax.bar(x + i * w, b.mean(), w, yerr=b.std(), capsize=3, label=m)

ax.set_xticks(x + w, [f'{l:.2f}' for l in VAL_LENS])
ax.set_xlabel('pole length')
ax.set_ylabel('val score')
ax.set_title('val score per length of the saved ckpt')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
fig.tight_layout()
plt.show()

In [ ]:
#where the curriculum puts its samples over training, mean over seeds
pcols = [f'p_{l:.2f}' for l in VAL_LENS]
p = load_val(groups['curriculum']).groupby('step')[pcols].mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.pcolormesh(p.index, VAL_LENS, p.T.values, shading='nearest', cmap='viridis')
fig.colorbar(im, ax=ax, label='sample prob')
ax.set_xlabel('env steps')
ax.set_ylabel('pole length')
ax.set_title('curriculum sampling probs per length bin')
plt.show()

In [ ]:
#training cost and when the saved ckpt was found
rows = []
for m in METHODS:
    for n in groups[m]:
        df = pd.read_csv(f'runs/{n}/val.csv')
        b = df.loc[df['score'].idxmax()]
        rows.append({'method': m, 'best_score': b['score'], 'best_step': b['step'],
                     'wall_min': df['time'].iloc[-1] / 60, 'val_steps': df['val_steps'].iloc[-1]})
pd.DataFrame(rows).groupby('method').agg(['mean', 'std']).round(3)